# 03 · Validate the two abstract models — Tutorial

This notebook walks through how we confirm the two models the abstract compares
work: **`switching_observer`** (the paper's model) and **`hb_integration`** (our
hierarchical Bayesian model).

"Does it work" has **three** separate meanings, each needing its own evidence:

1. **Correctness** — does the code compute what the equations say? *(verification suites)*
2. **Fittability** — can it be fit to a real subject and return sensible parameters? *(smoke fit)*
3. **Parameter recovery** — if we simulate data from known parameters and refit, do we get them back? *(the test that tells us a fitted number is trustworthy)*

Every step below runs end-to-end. After the important ones, a **Question** asks
you to interpret what you just saw — jot your answer, then check the solutions
notebook.

All of these checks call scripts that already live in the `observers/` package —
nothing here is hand-written analysis. You can run each from a terminal too
(the equivalent command is shown).

## Setup

Point Python at the package and confirm the data + models import.

In [ ]:
import sys, os
# Point Python at the package. Run this notebook from the repo root
# (hierarchical/) OR set the path below to your checkout.
REPO = os.path.abspath(os.getcwd())
if "observers" not in os.listdir(REPO):
    # adjust if you launched the notebook from docs/ or notebooks/
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO)
os.chdir(REPO)
print("repo root:", REPO)
print("observers present:", "observers" in os.listdir(REPO))

repo root: /Users/vestige/code/heartwood/nma-prep/group-project/hierarchical
observers present: True


Confirm all the pieces import.

In [ ]:
from observers.models.switching_observer import SwitchingObserver
from observers.models.hb_integration import HBIntegrationObserver
import observers.verification.verify_switching as verify_switch
import observers.verification.verify_hb_integration as verify_hb
import observers.fitting.switching_observer_fit as switch_fit
import observers.fitting.hb_integration_fit as hb_fit
import observers.fitting.online_recovery as recovery
print("all imports OK")

all imports OK


## Level 1 — Correctness

Each model ships a **verification suite**. These are stronger than "does it run":
each check compares the model against an **oracle with a known answer** (a
reduction, a limit, or an exact identity). If a reduction is off, the code is
wrong no matter how well it later fits.

Terminal equivalent:
```
python -m observers.verification.verify_switching
python -m observers.verification.verify_hb_integration
```

### switching_observer

In [ ]:
verify_switch.run()

=== Verification: static Switching observer ===
[PASS] evidence read-out == V(.;d,k_like)  — max|Δ|=1.21e-17, argmax=85 (want 85)
[PASS] prior read-out == delta at 225  — mass@225=1.0000, total=1.0000, argmax=225
[PASS] switch weights match Eq.6 reliability ratio  — P(evidence)=0.6667 (want 0.6667)
[PASS] all 108 condition distributions are valid  — invalid distributions = 0
[PASS] NLL lower at generating than at bad params  — NLL(true)=975.4 < NLL(bad)=1417.5
5/5 checks passed  ✅ ALL PASS


> **Question.** The third check says `switch weights match Eq.6 reliability ratio P(evidence)=0.6667`. Where does 0.6667 come from, given the prior and sensory concentrations used in that test?

### hb_integration

In [ ]:
verify_hb.run()

=== Verification: Hierarchical Bayesian INTEGRATION observer ===

-- T1 reduction --
[PASS] alpha=1 == Girshick (k_e=1.0, kappa=2.7)  — max|Δ|=2.78e-17
[PASS] alpha=1 == Girshick (k_e=8.0, kappa=8.7)  — max|Δ|=1.39e-17
[PASS] alpha=1 == Girshick (k_e=3.0, kappa=0.5)  — max|Δ|=0.00e+00

-- T2 normalisation --
[PASS] estimate distributions sum to 1  — max|sum-1|=2.22e-16

-- T3 emergence of bimodality --
[PASS] far+low-coh: bimodal (stimulus AND prior peaks)  — peaks=[67, 223]
[PASS] near-prior: unimodal  — peaks=[225]
[PASS] far+high-coh: evidence dominates (peak at stimulus)  — argmax=85, peaks=[85]

-- T4 alpha->0 --
[PASS] alpha->0: unimodal at the stimulus (prior gone)  — argmax=85, peaks=[85]

-- T5 discriminator vs switch --
    stim dirs [165, 145, 125, 105] (dist 60..120 from 225)
    integration prior-window mass: 0.062, 0.010, 0.002, 0.001   (CV=1.34)
    switch      prior-window mass: 0.266, 0.243, 0.240, 0.240   (CV=0.04)
[PASS] DISCRIMINATOR: integration prior-reliance decl

> **Question.** Two checks matter most here. (a) The `alpha=1 == Girshick` reduction passes at max|Δ| ~ 1e-17. Why is that reduction the single most important correctness check for this model? (b) The **discriminator** check reports `integ CV=1.34 vs switch CV=0.04`. What does that prove, and why does the whole abstract depend on it?

## Level 2 — Fittability

Can each model be fit to a real subject and return sensible parameters? We fit
**subject 1** (n = 8562 trials). The switch fits in ~30 s; HB integration is
slower (the read-out grid is evaluated per kappa), so we **cap the iterations**
here to keep the tutorial responsive — a production fit uses more starts and
runs 3–5 min/subject.

Terminal equivalent:
```
python -m observers.fitting.switching_observer_fit human 1
python -m observers.fitting.hb_integration_fit human 1
```

### Fit the switch to subject 1

In [ ]:
d = switch_fit._load_subject(1)
obs, nll, aic, bic = switch_fit.fit(d, maxiter=400)
print(f"switch subj1  n={len(d['estimates'])}  NLL={nll:.1f}  AIC={aic:.1f}  BIC={bic:.1f}")
print("implied prior SDs (deg):", {k: round(v,1) for k,v in obs.prior_sd_degrees().items()})
print("fitted k_like:", {k: round(float(v),2) for k,v in obs.k_like.items()})

switch subj1  n=8562  NLL=38869.7  AIC=77757.4  BIC=77820.9
implied prior SDs (deg): {'80': 96.4, '40': 78.3, '20': 36.5, '10': 8.2}
fitted k_like: {0.06: 1.0, 0.12: 15.71, 0.24: 145343.38}


> **Question.** The implied prior SDs come back around 96/78/36/8 deg. The experiment's four blocks had prior SDs of 80/40/20/10 deg. Is the model working? And why might `k_like[0.24]` come back as a huge number (~10^5)?

### Fit HB integration to subject 1 (capped)

In [ ]:
d = hb_fit._load_subject(1)
obs, nll, theta = hb_fit.fit(d, maxiter=60)   # capped smoke fit
import numpy as np
N=7; aic = 2*N + 2*nll
print(f"hb_integration subj1  NLL={nll:.1f}  AIC={aic:.1f}  (capped maxiter=60)")
print("fitted alpha (mixture weight):", round(float(obs.alpha),3))
print("fitted k_like:", {k: round(float(v),2) for k,v in obs.k_like.items()})

hb_integration subj1  NLL=38950.4  AIC=77914.8  (capped maxiter=60)
fitted alpha (mixture weight): 0.532
fitted k_like: {0.06: 1.0, 0.12: 10.03, 0.24: 49.27}


> **Question.** What does the fitted `alpha` (~0.5) mean in words, and why is it the parameter you most want to be able to trust?

## Level 3 — Parameter recovery (the decisive test)

Verification proves the code is correct; fitting proves it runs. **Recovery**
proves a fitted number is *trustworthy*: simulate data from known parameters,
refit, and check you get them back. If recovery fails, no parameter you report
from real data means anything.

Terminal equivalent:
```
python -c "from observers.fitting.online_recovery import static_parameter_recovery; static_parameter_recovery()"
python -m observers.fitting.hb_integration_fit recover
```

### Switch recovery

In [ ]:
ok_switch = recovery.static_parameter_recovery(seeds=(1,2,3))


=== parameter recovery (static switch) ===
  seed 1: k_like=(1.00,2.46,6.46) k_prior=(0.69,1.40,1.67,13.52)
  seed 2: k_like=(1.00,2.97,8.58) k_prior=(0.49,1.89,3.79,10.30)
  seed 3: k_like=(0.97,4.98,10.06) k_prior=(0.02,1.66,4.05,12.26)
  --- recovered mean vs truth ---
    k06      : recovered    0.992  truth   1.000  rel 0.01  [ok]
    k12      : recovered    3.468  truth   3.000  rel 0.16  [ok]
    k24      : recovered    8.367  truth   8.000  rel 0.05  [ok]
    kp80     : recovered    0.400  truth   0.500  rel 0.20  [ok]
    kp40     : recovered    1.650  truth   1.400  rel 0.18  [ok]
    kp20     : recovered    3.170  truth   2.700  rel 0.17  [ok]
    kp10     : recovered   12.026  truth   8.700  rel 0.38  [ok]
    k_motor  : recovered   40.129  truth  40.000  rel 0.00  [ok]
    p_random : recovered    0.022  truth   0.020  rel 0.08  [ok]
  -> static switch parameters recover (all within tolerance)


> **Question.** `k_like` recovers almost exactly, but `k_prior[10]` (the tightest prior, SD~10 deg) recovers least precisely. Why is the tight prior the hardest parameter to pin down?

### HB integration recovery

In [ ]:
hb_fit.recover()

=== parameter recovery (integration model) ===
  seed 1: alpha=0.600 p_rand=0.033 lam=0.043 k=(1.00,4.06,9.61) k_motor=35.4
  seed 2: alpha=0.594 p_rand=0.029 lam=0.027 k=(1.00,3.09,8.25) k_motor=30.4
  seed 3: alpha=0.610 p_rand=0.024 lam=0.067 k=(1.00,2.89,7.61) k_motor=31.7
  --- recovered mean vs truth ---
    k06      : recovered    1.000  truth   1.000  rel 0.00  [ok]
    k12      : recovered    3.345  truth   3.000  rel 0.11  [ok]
    k24      : recovered    8.490  truth   8.000  rel 0.06  [ok]
    alpha    : recovered    0.602  truth   0.600  rel 0.00  [ok]
    k_motor  : recovered   32.516  truth  30.000  rel 0.08  [ok]
    p_random : recovered    0.029  truth   0.030  rel 0.04  [ok]
    lam      : recovered    0.045  truth   0.050  rel 0.09  [ok]
  alpha identifiability: recovered alpha=0.602±0.006 (truth 0.6), p_random=0.029±0.004 (truth 0.03)
  -> alpha CLUSTERS near truth (identifiable)


> **Question.** The output ends with `alpha CLUSTERS near truth (identifiable)` and reports alpha and p_random separately. Why is checking alpha *against the lapse rate* the crucial part of this test?

## Summary

| | switching_observer | hb_integration |
|---|---|---|
| Correctness (verification) | 5/5 PASS | 12/12 PASS |
| Fittability (real subject) | ✓ | ✓ |
| Parameter recovery | ✓ (k_like tight; k_prior ordered) | ✓ (all <~10%; alpha identifiable) |

Both models are **verified, fittable, and identifiable**. Every check above is a
script in `observers/` you can re-run.

**Next (Task 5):** fit both models to all 12 subjects under identical
preprocessing and compare NLL / AIC / BIC plus cross-validated held-out
likelihood — the comparison that actually decides between them.